
# 29A — V5 CONFIRM2 Fresh Expansion Pool + Adaptive Batch Freeze

CONFIRM1 was **INCONCLUSIVE solely because evidence was insufficient**:
- PRIMARY pairable subjects = 8
- frozen minimum = 30
- V5 / Production Control scoring was never performed.

Therefore this notebook does **not** alter the model or the confirmation target.

It freezes a completely fresh CONFIRM2 pool of up to 320 exact-DOB identities,
with the original CONFIRM axis mixture scaled 4x:

- Competitive 80
- Project 100
- Status 140

It also freezes 16 deterministic batches of 20.

### Efficiency rule
Research batches in frozen order. After each **complete batch event freeze**, compute
only cumulative PRIMARY pairable-subject count (CONFIRM1 + completed CONFIRM2 batches).

Stop further research at the first batch boundary where cumulative PRIMARY pairable subjects >= 30.

No V5 score, Control score, chronology share, or axis performance may be inspected before that stop.


In [5]:

from pathlib import Path
from datetime import datetime
import ast, hashlib, json, re, time, unicodedata
import urllib.parse, urllib.request
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_VERSION="SAJU_ML_V5_CONFIRM2_POOL_FREEZE_29A_20260817"
SEED=2026081706
TARGET={"COMPETITIVE":80,"PROJECT":100,"STATUS":140}
FEMALE_MIN_SHARE=0.20
AXES=["COMPETITIVE","PROJECT","STATUS"]

def repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside Chartpalja repository.")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

def norm_name(x):
    s=unicodedata.normalize("NFKD",str(x))
    s="".join(c for c in s if not unicodedata.combining(c))
    s=s.casefold()
    return re.sub(r"[^a-z0-9]+","",s)

def is_female(x):
    return str(x).strip().lower().startswith("female")

def det_key(axis,qid,name):
    raw=f"{SEED}|{axis}|{qid}|{norm_name(name)}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

ROOT=repo_root()

C1=ROOT/"research/ml/artifacts/v5_confirm_final_evaluation"
C1_FINAL=C1/"V5_CONFIRM_FINAL_DECISION.json"
C1_PAIR=C1/"V5_CONFIRM_PAIRABILITY_DECISION.json"
C1_FREEZE=C1/"V5_CONFIRM_EVENT_FREEZE_DECISION.json"

AUDIT=ROOT/"research/ml/artifacts/v5_identity_linkage_audit"
POOL=AUDIT/"V5_IDENTITY_LINKAGE_AUDIT_CANDIDATE_POOL.csv"

REPAIR=ROOT/"research/ml/artifacts/v5_identity_repair"
DEV0=REPAIR/"V5_DEV_SUBJECT_ROSTER_160_REPAIRED.csv"
CONF1=REPAIR/"V5_CONFIRM_SUBJECT_ROSTER_80_REPAIRED_SEALED.csv"

E1=ROOT/"research/ml/artifacts/v5_dev_expansion_e1/V5_DEV_EXPANSION_E1_ROSTER_160.csv"
E2=ROOT/"research/ml/artifacts/v5_dev_expansion_e2/V5_DEV_EXPANSION_E2_ROSTER_160.csv"
E1_SUPP=ROOT/"research/ml/artifacts/v5_dev_expansion_e1/V5_DEV_EXPANSION_E1_STATUS_SUPPLEMENT_EXACT_DOB_POOL.csv"

BIRTH=ROOT/"research/ml/artifacts/v4_unified_dev_roster/PersonList-15k.csv"
CAND_SPEC=ROOT/"research/ml/artifacts/v5_tg10_multitask_balanced/V5_25A_FROZEN_CANDIDATE_MODEL_SPEC.json"
CAND_COEF=ROOT/"research/ml/artifacts/v5_tg10_multitask_balanced/V5_25A_FROZEN_CANDIDATE_COEFFICIENTS.csv"
CONTROL_DEC=ROOT/"research/ml/artifacts/v5_candidate_vs_control/V5_CANDIDATE_VS_PRODUCTION_CONTROL_DECISION.json"
CONF_PROTOCOL=ROOT/"research/ml_corpus/v5_ground_truth/V5_CONFIRM_ONE_SHOT_EVALUATION_PROTOCOL.json"
C2_PROTOCOL=ROOT/"research/ml_corpus/v5_ground_truth/V5_CONFIRM2_ADAPTIVE_EXPANSION_PROTOCOL.json"

OUT=ROOT/"research/ml/artifacts/v5_confirm2_expansion"
BATCH=OUT/"batches"
CACHE=OUT/"status_supplement_query_cache"
OUT.mkdir(parents=True,exist_ok=True)
BATCH.mkdir(parents=True,exist_ok=True)
CACHE.mkdir(parents=True,exist_ok=True)

required=[C1_FINAL,C1_PAIR,C1_FREEZE,POOL,DEV0,CONF1,E1,E2,BIRTH,
          CAND_SPEC,CAND_COEF,CONTROL_DEC,CONF_PROTOCOL,C2_PROTOCOL]
for p in required:
    if not p.exists(): raise FileNotFoundError(p)

c1=json.load(open(C1_FINAL,encoding="utf-8"))
c1pair=json.load(open(C1_PAIR,encoding="utf-8"))
c2proto=json.load(open(C2_PROTOCOL,encoding="utf-8"))
spec=json.load(open(CAND_SPEC,encoding="utf-8"))
ctrl=json.load(open(CONTROL_DEC,encoding="utf-8"))

assert c1["status"]=="V5_CONFIRM_INSUFFICIENT_EVIDENCE_NO_PRODUCTION_DECISION"
assert c1["classification"]=="INCONCLUSIVE"
assert c1["scoring_performed"] is False
assert int(c1["primary_pairable_subjects"])==8
assert int(c1["frozen_min_pairable_subjects"])==30
assert c1["rules"]["candidate_retuned"] is False
assert c1["rules"]["control_retuned"] is False
assert c1["rules"]["confirm_outcomes_used_for_model_fitting"] is False

assert c1pair["ready_to_score"] is False
assert c1pair["rules"]["astrology_scored_yet"] is False
assert c1pair["rules"]["control_scored_yet"] is False

assert c2proto["status"]=="PREDECLARED_AFTER_CONFIRM1_INSUFFICIENT_BEFORE_CONFIRM2_MEMBERSHIP_OR_EVENT_RESEARCH"
assert c2proto["confirm2_pool"]["axis_quotas"]==TARGET
assert spec["architecture"]=="TG10_MULTITASK_EFFECT_RIDGE_BALANCED"
assert spec["coefficients_sha256"]==sha256_file(CAND_COEF)
assert ctrl["status"]=="V5_FROZEN_CANDIDATE_BEATS_CONTROL_READY_FOR_CONFIRM_PROTOCOL"
assert sha256_file(ROOT/"saju_engine.py")==ctrl["lineage"]["saju_engine_py_sha256"]

print("29A PRE-FREEZE LINEAGE PASS")
print("CONFIRM1 scores remain unseen.")


29A PRE-FREEZE LINEAGE PASS
CONFIRM1 scores remain unseen.


## 1. Build exhaustive prior-identity exclusion set

In [6]:

used_frames=[]
for path,label in [(DEV0,"V5_DEV0"),(CONF1,"CONFIRM1"),(E1,"V5_E1"),(E2,"V5_E2")]:
    z=pd.read_csv(path)
    z["_origin"]=label
    used_frames.append(z)

# Also exclude older V4 rosters if present, preventing deep STATUS queries from reintroducing old research identities.
for path,label in [
    (ROOT/"research/ml/artifacts/v4_unified_dev_roster/V4_UNIFIED_DEV_SUBJECT_ROSTER_100.csv","V4_W1"),
    (ROOT/"research/ml/artifacts/v4_unified_dev_wave2_roster/V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_44.csv","V4_W2"),
]:
    if path.exists():
        z=pd.read_csv(path); z["_origin"]=label; used_frames.append(z)

used=pd.concat(used_frames,ignore_index=True,sort=False)
used_names={norm_name(x) for x in used["name"].dropna()}
used_qids=set()
if "wikidata_id" in used.columns:
    used_qids={str(x) for x in used["wikidata_id"].dropna() if str(x).startswith("Q")}

assert len(used_names)>=560, "Unexpectedly small used identity exclusion set."

print("Used normalized names blocked:",len(used_names))
print("Used Wikidata QIDs blocked:",len(used_qids))


Used normalized names blocked: 692
Used Wikidata QIDs blocked: 560


## 2. Base fresh PASS_EXACT_DOB candidate pool

In [7]:

pool=pd.read_csv(POOL)
required_cols={"axis","linkage_status","wikidata_id","name","birth_date","birth_time",
               "utc_offset","birth_place","latitude","longitude","source_row_key","gender"}
missing=required_cols-set(pool.columns)
if missing:
    raise RuntimeError(f"Candidate pool missing columns: {sorted(missing)}")

cand=pool[
    (pool.linkage_status=="PASS_EXACT_DOB")
    & pool.wikidata_id.notna()
    & pool.axis.isin(AXES)
].copy()

cand["norm_name_c2"]=cand.name.map(norm_name)
cand=cand[
    ~cand.norm_name_c2.isin(used_names)
    & ~cand.wikidata_id.astype(str).isin(used_qids)
].copy()

cand["candidate_source_c2"]="AUDITED_PASS_EXACT_DOB_POOL"
cand["linkage_status"]="PASS_EXACT_DOB"

cand=(
    cand.sort_values(["axis","wikidata_id","norm_name_c2"])
    .drop_duplicates("wikidata_id")
    .drop_duplicates(["axis","norm_name_c2"])
    .reset_index(drop=True)
)

print("Base fresh exact-DOB supply:",cand.axis.value_counts().to_dict())


Base fresh exact-DOB supply: {'PROJECT': 745, 'COMPETITIVE': 15, 'STATUS': 15}


## 3. Reuse / regenerate frozen-role STATUS supplement only if supply is short

In [9]:
# ================================================================
# 3. Reuse / regenerate frozen-role axis supplements if supply short
#
# R2 supply-only patch:
# - STATUS: existing frozen supplement logic preserved
# - COMPETITIVE: if fresh exact-DOB supply is insufficient,
#   deepen ONLY the ORIGINAL V5 competitive role taxonomy:
#     athlete / association football / basketball / tennis / boxer
#
# IMPORTANT:
# - no events
# - no event years
# - no pairability
# - no chronology
# - no astrology
# - no Control
# - exact Wikidata P569 == unique Rodden-AA calendar DOB required
# ================================================================

STATUS_ROLES = c2proto[
    "status_supply"
]["supplement_roles_same_as_prior_predeclared_E1_status_supply"]

# These are the ORIGINAL V5 Competitive role QIDs.
# They are not selected after observing event results.
COMPETITIVE_ROLES = {
    "athlete": "Q2066131",
    "association_football_player": "Q937857",
    "basketball_player": "Q3665646",
    "tennis_player": "Q10833314",
    "boxer": "Q11338576",
}


def align_append(base, extra):
    if extra is None or len(extra) == 0:
        return base

    b = base.copy()
    e = extra.copy()

    for c in b.columns:
        if c not in e.columns:
            e[c] = np.nan

    for c in e.columns:
        if c not in b.columns:
            b[c] = np.nan

    e = e[b.columns]

    return pd.concat(
        [b, e],
        ignore_index=True,
        sort=False,
    )


# ----------------------------------------------------------------
# 3A. Reuse already-created STATUS exact-DOB supplement
# ----------------------------------------------------------------

def clean_status_extra(z, source_label):
    if z is None or len(z) == 0:
        return pd.DataFrame()

    z = z.copy()

    if "axis" not in z.columns:
        z["axis"] = "STATUS"

    if "linkage_status" not in z.columns:
        z["linkage_status"] = "PASS_EXACT_DOB"

    if "norm_name_c2" not in z.columns:
        z["norm_name_c2"] = z.name.map(norm_name)

    z = z[
        (z.axis == "STATUS")
        & (z.linkage_status == "PASS_EXACT_DOB")
        & z.wikidata_id.notna()
        & ~z.norm_name_c2.isin(used_names)
        & ~z.wikidata_id.astype(str).isin(used_qids)
    ].copy()

    z["candidate_source_c2"] = source_label

    return z


if E1_SUPP.exists():
    oldsupp = pd.read_csv(E1_SUPP)

    oldsupp = clean_status_extra(
        oldsupp,
        "E1_STATUS_SUPPLEMENT_REUSED",
    )

    cand = align_append(
        cand,
        oldsupp,
    )


cand = (
    cand
    .sort_values(
        [
            "axis",
            "candidate_source_c2",
            "wikidata_id",
            "norm_name_c2",
        ],
        kind="stable",
    )
    .drop_duplicates("wikidata_id")
    .drop_duplicates(
        ["axis", "norm_name_c2"]
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------------
# 3B. Build UNIQUE strict Rodden-AA birth lookup
#
# Needed only for fresh role-only exact-DOB linkage.
# ----------------------------------------------------------------

birth = pd.read_csv(BIRTH)

required_birth_cols = {
    "RowKey",
    "BirthTime",
    "Gender",
    "Name",
    "Notes",
}

if not required_birth_cols.issubset(
    set(birth.columns)
):
    raise RuntimeError(
        "Unexpected PersonList-15k schema: %s"
        % list(birth.columns)
    )


def parse_birth_blob(blob):
    obj = json.loads(str(blob))

    std = obj["StdTime"]
    loc = obj["Location"]

    m = re.match(
        r"^(\d{1,2}):(\d{2})\s+"
        r"(\d{2})/(\d{2})/(\d{4})\s+"
        r"([+-]\d{2}:\d{2})$",
        std.strip(),
    )

    if not m:
        raise ValueError(std)

    hh, mm, dd, mo, yyyy, offset = (
        m.groups()
    )

    return {
        "birth_date":
            "%04d-%02d-%02d"
            % (
                int(yyyy),
                int(mo),
                int(dd),
            ),
        "birth_time":
            "%02d:%02d"
            % (
                int(hh),
                int(mm),
            ),
        "utc_offset": offset,
        "birth_year": int(yyyy),
        "birth_place": loc["Name"],
        "longitude": float(
            loc["Longitude"]
        ),
        "latitude": float(
            loc["Latitude"]
        ),
    }


def rodden_grade(notes):
    s = str(notes)

    try:
        obj = ast.literal_eval(s)

        if isinstance(obj, dict):
            return str(
                obj.get("rodden") or ""
            ).strip().upper()

    except Exception:
        pass

    m = re.search(
        r"rodden['\"]?\s*:\s*"
        r"['\"]([^'\"]+)['\"]",
        s,
        flags=re.I,
    )

    return (
        m.group(1).strip().upper()
        if m
        else ""
    )


aa_rows = []

for _, r in birth.iterrows():

    if rodden_grade(
        r["Notes"]
    ) != "AA":
        continue

    try:
        p = parse_birth_blob(
            r["BirthTime"]
        )

    except Exception:
        continue

    if not (
        1900
        <= int(p["birth_year"])
        <= 1995
    ):
        continue

    aa_rows.append(
        {
            "source_row_key":
                r["RowKey"],
            "source_name":
                r["Name"],
            "gender":
                str(
                    r["Gender"]
                ).lower(),
            "norm_name_c2":
                norm_name(
                    r["Name"]
                ),
            **p,
        }
    )


aa = pd.DataFrame(
    aa_rows
)

name_n = (
    aa
    .groupby(
        "norm_name_c2"
    )
    .size()
)

ambiguous_names = set(
    name_n[
        name_n > 1
    ].index
)

aa_unique = aa[
    ~aa.norm_name_c2.isin(
        ambiguous_names
    )
].copy()

if len(aa_unique) < 500:
    raise RuntimeError(
        "Unexpectedly small "
        "strict-AA unique birth pool."
    )


print(
    "Unique strict-AA "
    "birth identities:",
    len(aa_unique),
)


# ----------------------------------------------------------------
# 3C. Shared role+DOB Wikidata query helper
# ----------------------------------------------------------------

ENDPOINT = (
    "https://query.wikidata.org/sparql"
)


def role_dob_query(
    qid,
    limit=20000,
):

    # P569 is used ONLY for exact identity linkage.
    #
    # No event / award / position result /
    # chronology variable is queried.

    return """
PREFIX wd:
    <http://www.wikidata.org/entity/>
PREFIX wdt:
    <http://www.wikidata.org/prop/direct/>
PREFIX rdfs:
    <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT
    ?person
    ?personLabel
    ?dob

WHERE {

  ?person
      wdt:P106 wd:%s ;
      wdt:P569 ?dob ;
      rdfs:label ?personLabel .

  FILTER(
      LANG(?personLabel) = "en"
  )

  FILTER(
      YEAR(?dob) >= 1900
      &&
      YEAR(?dob) <= 1995
  )
}

ORDER BY ?person
LIMIT %d
""" % (
        qid,
        int(limit),
    )


def run_wdqs(
    query,
    cache_path,
    max_attempts=4,
):

    if cache_path.exists():
        return json.load(
            open(
                cache_path,
                encoding="utf-8",
            )
        )

    params = urllib.parse.urlencode(
        {
            "query": query,
            "format": "json",
        }
    )

    url = (
        ENDPOINT
        + "?"
        + params
    )

    headers = {
        "Accept":
            "application/"
            "sparql-results+json",

        "User-Agent":
            "Chartpalja-Saju-Research/"
            "1.0 "
            "(CONFIRM2 role-only "
            "exact-DOB candidate supply)",
    }

    last = None

    for attempt in range(
        max_attempts
    ):

        try:

            req = (
                urllib.request.Request(
                    url,
                    headers=headers,
                )
            )

            with urllib.request.urlopen(
                req,
                timeout=120,
            ) as resp:

                obj = json.loads(
                    resp
                    .read()
                    .decode("utf-8")
                )

            json.dump(
                obj,
                open(
                    cache_path,
                    "w",
                    encoding="utf-8",
                ),
                ensure_ascii=False,
            )

            return obj

        except Exception as e:

            last = e

            if (
                attempt + 1
                < max_attempts
            ):

                sleep_s = (
                    3
                    * (
                        2 ** attempt
                    )
                )

                print(
                    "WDQS retry",
                    sleep_s,
                    "sec:",
                    repr(e),
                )

                time.sleep(
                    sleep_s
                )

    raise RuntimeError(
        "WDQS failed for "
        f"{cache_path.name}: "
        f"{last!r}"
    )


def query_and_resolve_axis_roles(
    axis,
    role_map,
    cache_prefix,
    candidate_source,
):

    rows = []
    query_manifest = {}

    for role_name, qid in (
        role_map.items()
    ):

        cache_path = (
            CACHE
            / (
                f"{cache_prefix}"
                f"__{role_name}.json"
            )
        )

        print(
            "supplement query:",
            axis,
            role_name,
            qid,
        )

        obj = run_wdqs(
            role_dob_query(
                qid
            ),
            cache_path,
        )

        bindings = (
            obj
            .get(
                "results",
                {},
            )
            .get(
                "bindings",
                [],
            )
        )

        query_manifest[
            role_name
        ] = {
            "qid": qid,
            "returned_rows":
                len(bindings),
            "cache_file":
                cache_path.name,
            "cache_sha256":
                sha256_file(
                    cache_path
                ),
        }

        print(
            "  returned:",
            len(bindings),
        )

        for b in bindings:

            uri = (
                b.get("person")
                or {}
            ).get(
                "value",
                "",
            )

            label = (
                b.get(
                    "personLabel"
                )
                or {}
            ).get(
                "value",
                "",
            )

            dob = (
                b.get("dob")
                or {}
            ).get(
                "value",
                "",
            )

            qid_person = (
                uri.rsplit(
                    "/",
                    1,
                )[-1]
                if uri
                else ""
            )

            m = re.match(
                r"^(\d{4})-"
                r"(\d{2})-"
                r"(\d{2})",
                dob,
            )

            if not (
                qid_person
                and label
                and m
            ):
                continue

            yyyy, mo, dd = map(
                int,
                m.groups(),
            )

            # Reject low-precision dates.
            if (
                mo < 1
                or dd < 1
            ):
                continue

            rows.append(
                {
                    "axis": axis,
                    "role_family":
                        role_name,
                    "role_qid":
                        qid,
                    "wikidata_id":
                        qid_person,
                    "name":
                        label,
                    "norm_name_c2":
                        norm_name(
                            label
                        ),
                    "wikidata_birth_date":
                        "%04d-%02d-%02d"
                        % (
                            yyyy,
                            mo,
                            dd,
                        ),
                }
            )

    raw = pd.DataFrame(
        rows
    )

    if len(raw) == 0:
        raise RuntimeError(
            f"{axis} supplement "
            "returned no rows."
        )

    raw = (
        raw
        .sort_values(
            [
                "wikidata_id",
                "role_family",
            ]
        )
        .drop_duplicates(
            [
                "wikidata_id",
                "norm_name_c2",
                "wikidata_birth_date",
            ]
        )
        .reset_index(drop=True)
    )

    # ------------------------------------------------------------
    # CRITICAL IDENTITY RULE
    #
    # English normalized name
    # AND exact day-level DOB
    # must match the frozen AA snapshot.
    # ------------------------------------------------------------

    resolved = raw.merge(
        aa_unique,
        left_on=[
            "norm_name_c2",
            "wikidata_birth_date",
        ],
        right_on=[
            "norm_name_c2",
            "birth_date",
        ],
        how="inner",
        validate="many_to_one",
    )

    # Absolutely no prior research identity
    # can re-enter CONFIRM2.
    resolved = resolved[
        ~resolved
        .norm_name_c2
        .isin(
            used_names
        )
        &
        ~resolved
        .wikidata_id
        .astype(str)
        .isin(
            used_qids
        )
    ].copy()

    resolved = (
        resolved
        .sort_values(
            [
                "wikidata_id",
                "role_family",
                "source_row_key",
            ]
        )
        .drop_duplicates(
            "wikidata_id"
        )
        .drop_duplicates(
            "norm_name_c2"
        )
        .reset_index(
            drop=True
        )
    )

    resolved[
        "linkage_status"
    ] = "PASS_EXACT_DOB"

    resolved[
        "candidate_source_c2"
    ] = candidate_source

    resolved[
        "rodden_rating"
    ] = "AA"

    resolved[
        "birth_source"
    ] = (
        "Frozen VedAstro "
        "PersonList-15k snapshot"
    )

    resolved[
        "selection_information_used"
    ] = (
        "ROLE_ONLY + "
        "WIKIDATA_P569_"
        "EXACT_DOB_LINKAGE"
    )

    resolved[
        "event_collection_started"
    ] = False

    resolved[
        "astrology_scored"
    ] = False

    return (
        resolved,
        query_manifest,
    )


# ----------------------------------------------------------------
# 3D. STATUS supply
#
# Usually already sufficient from reused E1 supplement.
# Query only if actually below the already-frozen target.
# ----------------------------------------------------------------

status_supply = int(
    (
        cand.axis
        == "STATUS"
    ).sum()
)

print(
    "STATUS after reusable "
    "supplement:",
    status_supply,
)


if (
    status_supply
    < TARGET["STATUS"]
):

    print(
        "STATUS exact-DOB "
        "supply shortage:",
        status_supply,
        "/",
        TARGET["STATUS"],
    )

    resolved_status, status_manifest = (
        query_and_resolve_axis_roles(
            axis="STATUS",
            role_map=STATUS_ROLES,
            cache_prefix="STATUS",
            candidate_source=(
                "C2_STATUS_ROLE_"
                "SUPPLEMENT_EXACT_DOB"
            ),
        )
    )

    status_out = (
        OUT
        / (
            "V5_CONFIRM2_STATUS_"
            "SUPPLEMENT_EXACT_DOB_"
            "POOL.csv"
        )
    )

    resolved_status.to_csv(
        status_out,
        index=False,
    )

    json.dump(
        {
            "version":
                "V5_CONFIRM2_STATUS_"
                "SUPPLEMENT_QUERY_"
                "MANIFEST_V1",

            "status":
                "CONFIRM2_STATUS_"
                "SUPPLEMENT_"
                "EXACT_DOB_LINKED",

            "roles":
                status_manifest,

            "resolved_fresh_n":
                int(
                    len(
                        resolved_status
                    )
                ),

            "birth_snapshot_sha256":
                sha256_file(BIRTH),

            "supplement_pool_sha256":
                sha256_file(
                    status_out
                ),

            "events_used": False,
            "pairability_used": False,
            "chronology_used": False,
            "astrology_used": False,
            "control_used": False,
        },
        open(
            OUT
            / (
                "V5_CONFIRM2_STATUS_"
                "SUPPLEMENT_QUERY_"
                "MANIFEST.json"
            ),
            "w",
            encoding="utf-8",
        ),
        ensure_ascii=False,
        indent=2,
    )

    cand = align_append(
        cand,
        resolved_status,
    )


# ----------------------------------------------------------------
# 3E. COMPETITIVE supply patch — R2 robust incremental querying
#
# Technical patch only:
# - no event / pair / chronology / astrology / Control information
# - reuse successful caches first
# - resolve exact-DOB candidates after EACH role
# - stop querying as soon as frozen C2 Competitive supply >= 80
# - huge association-football query is never sent as one monolith;
#   it is queried in 5-year DOB shards only if still needed.
# ----------------------------------------------------------------

competitive_supply = int((cand.axis == "COMPETITIVE").sum())

if competitive_supply < TARGET["COMPETITIVE"]:

    COMPETITIVE_ROLE_PLAN = [
        ("athlete", "Q2066131"),
        ("tennis_player", "Q10833314"),
        ("boxer", "Q11338576"),
        ("basketball_player", "Q3665646"),
    ]
    FOOTBALL_ROLE = ("association_football_player", "Q937857")

    patch_decision = {
        "version": "V5_CONFIRM2_COMPETITIVE_SUPPLY_PATCH_R2",
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "status": (
            "PREDECLARED_AFTER_WDQS_MONOLITHIC_QUERY_FAILURE_"
            "BEFORE_R2_INCREMENTAL_SUPPLEMENT_QUERY"
        ),
        "observed_supply_fact": {
            "fresh_PASS_EXACT_DOB_before_supplement": competitive_supply,
            "frozen_target": int(TARGET["COMPETITIVE"]),
        },
        "technical_failure_observed": (
            "Monolithic association_football_player WDQS query returned "
            "502/504 gateway errors before any event/model information was used."
        ),
        "remedy": {
            "query_small_or_cached_original_roles_incrementally": [
                x[0] for x in COMPETITIVE_ROLE_PLAN
            ],
            "association_football_player_query_mode": "5_YEAR_DOB_SHARDS_ONLY_IF_NEEDED",
            "stop_querying_when_fresh_competitive_supply_ge": int(
                TARGET["COMPETITIVE"]
            ),
        },
        "role_taxonomy_changed": False,
        "identity_rule": (
            "unique frozen Rodden-AA normalized name + exact Wikidata "
            "day-level P569 DOB match"
        ),
        "forbidden": {
            "events": True,
            "event_years": True,
            "pairability": True,
            "pair_gap": True,
            "chronology": True,
            "astrology": True,
            "control": True,
        },
        "target_reduction_allowed": False,
    }

    patch_path = OUT / "V5_CONFIRM2_COMPETITIVE_SUPPLY_PATCH_DECISION.json"
    json.dump(
        patch_decision,
        open(patch_path, "w", encoding="utf-8"),
        ensure_ascii=False,
        indent=2,
    )

    print(
        "COMPETITIVE exact-DOB supply shortage:",
        competitive_supply,
        "/",
        TARGET["COMPETITIVE"],
    )
    print("Running R2 incremental ORIGINAL-role Competitive supplement.")

    C2_WDQS_ENDPOINT = "https://query.wikidata.org/sparql"

    def _role_dob_query_range(qid, year_lo=1900, year_hi=1995, limit=25000):
        return """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?person ?personLabel ?dob WHERE {
  ?person wdt:P106 wd:%s ;
          wdt:P569 ?dob ;
          rdfs:label ?personLabel .

  FILTER(LANG(?personLabel) = "en")
  FILTER(YEAR(?dob) >= %d && YEAR(?dob) <= %d)
}
ORDER BY ?person
LIMIT %d
""" % (qid, int(year_lo), int(year_hi), int(limit))

    def _safe_wdqs(query, cache_path, max_attempts=4):
        # Successful output from the previous failed run (e.g. athlete)
        # is reused deterministically.
        if cache_path.exists():
            print("  cache HIT:", cache_path.name)
            return json.load(open(cache_path, encoding="utf-8"))

        params = urllib.parse.urlencode({
            "query": query,
            "format": "json",
        })
        url = C2_WDQS_ENDPOINT + "?" + params

        headers = {
            "Accept": "application/sparql-results+json",
            "User-Agent": (
                "Chartpalja-Saju-Research/1.0 "
                "(CONFIRM2 outcome-blind exact-DOB candidate supply)"
            ),
        }

        last = None
        for attempt in range(max_attempts):
            try:
                req = urllib.request.Request(url, headers=headers)
                with urllib.request.urlopen(req, timeout=120) as resp:
                    obj = json.loads(resp.read().decode("utf-8"))

                json.dump(
                    obj,
                    open(cache_path, "w", encoding="utf-8"),
                    ensure_ascii=False,
                )
                return obj

            except Exception as e:
                last = e
                if attempt + 1 < max_attempts:
                    sleep_s = 3 * (2 ** attempt)
                    print("  WDQS retry", sleep_s, "sec:", repr(e))
                    time.sleep(sleep_s)

        # Do not poison the whole study because one nonessential role endpoint
        # failed. Caller can move to the next frozen role / smaller shard.
        print("  WDQS role/shard unavailable:", cache_path.name, repr(last))
        return None

    def _resolve_competitive_obj(obj, role_name, qid, source_label):
        if obj is None:
            return pd.DataFrame()

        rows = []
        bindings = obj.get("results", {}).get("bindings", [])
        print("  returned:", len(bindings))

        for b in bindings:
            uri = (b.get("person") or {}).get("value", "")
            label = (b.get("personLabel") or {}).get("value", "")
            dob = (b.get("dob") or {}).get("value", "")

            qid_person = uri.rsplit("/", 1)[-1] if uri else ""
            m = re.match(r"^(\d{4})-(\d{2})-(\d{2})", dob)

            if not (qid_person and label and m):
                continue

            yyyy, mo, dd = map(int, m.groups())
            if mo < 1 or dd < 1:
                continue

            rows.append({
                "axis": "COMPETITIVE",
                "role_family": role_name,
                "role_qid": qid,
                "wikidata_id": qid_person,
                "name": label,
                "norm_name_c2": norm_name(label),
                "wikidata_birth_date": "%04d-%02d-%02d" % (
                    yyyy, mo, dd
                ),
            })

        if not rows:
            return pd.DataFrame()

        raw = pd.DataFrame(rows)
        raw = (
            raw.sort_values(["wikidata_id", "role_family"])
            .drop_duplicates([
                "wikidata_id",
                "norm_name_c2",
                "wikidata_birth_date",
            ])
            .reset_index(drop=True)
        )

        # Exact identity linkage: same normalized unique AA name AND same DOB.
        resolved = raw.merge(
            aa_unique,
            left_on=["norm_name_c2", "wikidata_birth_date"],
            right_on=["norm_name_c2", "birth_date"],
            how="inner",
            validate="many_to_one",
        )

        # Never re-enter any prior researched identity.
        resolved = resolved[
            ~resolved.norm_name_c2.isin(used_names)
            & ~resolved.wikidata_id.astype(str).isin(used_qids)
        ].copy()

        resolved = (
            resolved.sort_values([
                "wikidata_id",
                "role_family",
                "source_row_key",
            ])
            .drop_duplicates("wikidata_id")
            .drop_duplicates("norm_name_c2")
            .reset_index(drop=True)
        )

        if len(resolved) == 0:
            return resolved

        resolved["linkage_status"] = "PASS_EXACT_DOB"
        resolved["candidate_source_c2"] = source_label
        resolved["rodden_rating"] = "AA"
        resolved["birth_source"] = (
            "Frozen VedAstro PersonList-15k snapshot"
        )
        resolved["selection_information_used"] = (
            "ORIGINAL_COMPETITIVE_ROLE_ONLY + "
            "WIKIDATA_P569_EXACT_DOB_LINKAGE"
        )
        resolved["event_collection_started"] = False
        resolved["astrology_scored"] = False

        return resolved

    def _dedup_comp_parts(parts):
        good = [x for x in parts if x is not None and len(x)]
        if not good:
            return pd.DataFrame()

        z = pd.concat(good, ignore_index=True, sort=False)
        return (
            z.sort_values(
                ["wikidata_id", "role_family", "norm_name_c2"],
                kind="stable",
            )
            .drop_duplicates("wikidata_id")
            .drop_duplicates("norm_name_c2")
            .reset_index(drop=True)
        )

    def _supply_with_extra(base, extra):
        z = align_append(base, extra)
        if len(z) == 0:
            return 0

        z["norm_name_c2"] = z.name.map(norm_name)
        z = (
            z.sort_values(
                ["axis", "wikidata_id", "norm_name_c2"],
                kind="stable",
            )
            .drop_duplicates("wikidata_id")
            .drop_duplicates(["axis", "norm_name_c2"])
        )
        return int((z.axis == "COMPETITIVE").sum())

    comp_parts = []
    query_records = []

    # 1) Cached / smaller original roles first.
    # Athlete cache from the failed run should normally be reused here.
    for role_name, qid in COMPETITIVE_ROLE_PLAN:
        if _supply_with_extra(cand, _dedup_comp_parts(comp_parts)) >= TARGET["COMPETITIVE"]:
            break

        cache_path = CACHE / f"COMPETITIVE__{role_name}.json"
        print("supplement query:", "COMPETITIVE", role_name, qid)

        obj = _safe_wdqs(
            _role_dob_query_range(qid, 1900, 1995),
            cache_path,
        )

        res = _resolve_competitive_obj(
            obj,
            role_name,
            qid,
            "C2_COMPETITIVE_ORIGINAL_ROLE_SUPPLEMENT_EXACT_DOB",
        )

        if len(res):
            comp_parts.append(res)

        current_supply = _supply_with_extra(
            cand,
            _dedup_comp_parts(comp_parts),
        )

        query_records.append({
            "role_family": role_name,
            "role_qid": qid,
            "query_mode": "FULL_1900_1995",
            "cache_file": cache_path.name,
            "cache_sha256": (
                sha256_file(cache_path)
                if cache_path.exists()
                else None
            ),
            "resolved_exact_DOB_rows": int(len(res)),
            "fresh_competitive_supply_after": current_supply,
        })

        print(
            "  exact-DOB resolved:",
            len(res),
            "| cumulative Competitive supply:",
            current_supply,
        )

    # 2) Football is huge. Only if still needed, query deterministic
    #    5-year birth shards, stopping as soon as target is reached.
    if _supply_with_extra(
        cand,
        _dedup_comp_parts(comp_parts),
    ) < TARGET["COMPETITIVE"]:

        role_name, qid = FOOTBALL_ROLE
        print(
            "Football still needed; switching to deterministic 5-year DOB shards."
        )

        for year_lo in range(1900, 1996, 5):
            if _supply_with_extra(
                cand,
                _dedup_comp_parts(comp_parts),
            ) >= TARGET["COMPETITIVE"]:
                break

            year_hi = min(year_lo + 4, 1995)
            cache_path = CACHE / (
                f"COMPETITIVE__{role_name}"
                f"__{year_lo}_{year_hi}.json"
            )

            print(
                "supplement shard:",
                role_name,
                year_lo,
                "-",
                year_hi,
            )

            obj = _safe_wdqs(
                _role_dob_query_range(
                    qid,
                    year_lo,
                    year_hi,
                    limit=10000,
                ),
                cache_path,
            )

            res = _resolve_competitive_obj(
                obj,
                role_name,
                qid,
                "C2_COMPETITIVE_ORIGINAL_ROLE_SUPPLEMENT_EXACT_DOB",
            )

            if len(res):
                comp_parts.append(res)

            current_supply = _supply_with_extra(
                cand,
                _dedup_comp_parts(comp_parts),
            )

            query_records.append({
                "role_family": role_name,
                "role_qid": qid,
                "query_mode": "DOB_5Y_SHARD",
                "year_lo": year_lo,
                "year_hi": year_hi,
                "cache_file": cache_path.name,
                "cache_sha256": (
                    sha256_file(cache_path)
                    if cache_path.exists()
                    else None
                ),
                "resolved_exact_DOB_rows": int(len(res)),
                "fresh_competitive_supply_after": current_supply,
            })

            print(
                "  exact-DOB resolved:",
                len(res),
                "| cumulative Competitive supply:",
                current_supply,
            )

    resolved_comp = _dedup_comp_parts(comp_parts)

    final_comp_supply = _supply_with_extra(cand, resolved_comp)

    comp_out = OUT / (
        "V5_CONFIRM2_COMPETITIVE_SUPPLEMENT_EXACT_DOB_POOL.csv"
    )
    resolved_comp.to_csv(comp_out, index=False)

    comp_manifest_path = OUT / (
        "V5_CONFIRM2_COMPETITIVE_SUPPLEMENT_QUERY_MANIFEST.json"
    )
    json.dump(
        {
            "version": "V5_CONFIRM2_COMPETITIVE_SUPPLEMENT_QUERY_MANIFEST_R2",
            "status": (
                "CONFIRM2_COMPETITIVE_ORIGINAL_ROLE_"
                "INCREMENTAL_SUPPLEMENT_COMPLETE"
            ),
            "query_records": query_records,
            "resolved_fresh_unique_exact_DOB_n": int(len(resolved_comp)),
            "fresh_competitive_supply_after": final_comp_supply,
            "target": int(TARGET["COMPETITIVE"]),
            "target_reached": bool(
                final_comp_supply >= TARGET["COMPETITIVE"]
            ),
            "birth_snapshot_sha256": sha256_file(BIRTH),
            "supply_patch_sha256": sha256_file(patch_path),
            "supplement_pool_sha256": sha256_file(comp_out),
            "events_used": False,
            "pairability_used": False,
            "chronology_used": False,
            "astrology_used": False,
            "control_used": False,
        },
        open(comp_manifest_path, "w", encoding="utf-8"),
        ensure_ascii=False,
        indent=2,
    )

    if final_comp_supply < TARGET["COMPETITIVE"]:
        raise RuntimeError({
            "status": (
                "V5_CONFIRM2_COMPETITIVE_SUPPLY_STILL_INSUFFICIENT_"
                "AFTER_ORIGINAL_ROLE_INCREMENTAL_EXPANSION"
            ),
            "fresh_competitive_available": final_comp_supply,
            "target": int(TARGET["COMPETITIVE"]),
            "scores_viewed": False,
            "events_used_for_membership": False,
            "next_rule": (
                "Do not lower target. Freeze a separate outcome-blind "
                "Competitive taxonomy expansion before any new role query."
            ),
        })

    cand = align_append(cand, resolved_comp)

    print(
        "COMPETITIVE R2 supplement PASS:",
        final_comp_supply,
        ">=",
        TARGET["COMPETITIVE"],
    )

# ----------------------------------------------------------------
# 3F. Final conservative dedup + supply gate
# ----------------------------------------------------------------

cand[
    "norm_name_c2"
] = cand.name.map(
    norm_name
)

cand = cand[
    ~cand
    .norm_name_c2
    .isin(
        used_names
    )
    &
    ~cand
    .wikidata_id
    .astype(str)
    .isin(
        used_qids
    )
].copy()


source_priority = {
    "AUDITED_PASS_EXACT_DOB_POOL":
        0,

    "E1_STATUS_SUPPLEMENT_REUSED":
        1,

    "C2_STATUS_ROLE_SUPPLEMENT_EXACT_DOB":
        2,

    "C2_COMPETITIVE_ORIGINAL_ROLE_SUPPLEMENT_EXACT_DOB":
        2,
}


cand["_sp"] = (
    cand
    .candidate_source_c2
    .map(
        source_priority
    )
    .fillna(9)
)


cand = (
    cand
    .sort_values(
        [
            "axis",
            "_sp",
            "wikidata_id",
            "norm_name_c2",
        ],
        kind="stable",
    )
    .drop_duplicates(
        "wikidata_id"
    )
    .drop_duplicates(
        [
            "axis",
            "norm_name_c2",
        ]
    )
    .drop(
        columns="_sp"
    )
    .reset_index(
        drop=True
    )
)


# Every C2 identity must still
# satisfy the conservative linkage rule.
assert (
    cand.linkage_status
    == "PASS_EXACT_DOB"
).all()


avail = (
    cand
    .axis
    .value_counts()
    .to_dict()
)


supply_table = pd.DataFrame(
    [
        {
            "axis": a,

            "fresh_exact_DOB_available":
                int(
                    avail.get(
                        a,
                        0,
                    )
                ),

            "C2_target":
                int(
                    TARGET[a]
                ),

            "sufficient":
                (
                    int(
                        avail.get(
                            a,
                            0,
                        )
                    )
                    >=
                    int(
                        TARGET[a]
                    )
                ),
        }

        for a in AXES
    ]
)


display(
    supply_table
)


supply_table.to_csv(
    OUT
    / (
        "V5_CONFIRM2_"
        "FRESH_EXACT_DOB_"
        "SUPPLY_SUMMARY.csv"
    ),
    index=False,
)


if not (
    supply_table
    .sufficient
    .all()
):

    diag = {
        "status":
            "V5_CONFIRM2_"
            "CANDIDATE_SUPPLY_"
            "INSUFFICIENT_"
            "DO_NOT_LOWER_TARGET",

        "available":
            avail,

        "target":
            TARGET,

        "scores_viewed":
            False,

        "events_viewed_for_"
        "membership":
            False,

        "next_rule":
            (
                "Do not lower C2 quotas. "
                "Expand only outcome-blind "
                "role-based exact-DOB "
                "candidate supply."
            ),
    }

    json.dump(
        diag,
        open(
            OUT
            / (
                "V5_CONFIRM2_"
                "SUPPLY_"
                "INSUFFICIENT.json"
            ),
            "w",
            encoding="utf-8",
        ),
        ensure_ascii=False,
        indent=2,
    )

    raise RuntimeError(
        diag
    )


print(
    "CONFIRM2 fresh exact-DOB "
    "supply gate: PASS"
)

Unique strict-AA birth identities: 12795
STATUS after reusable supplement: 169
COMPETITIVE exact-DOB supply shortage: 15 / 80
Running R2 incremental ORIGINAL-role Competitive supplement.
supplement query: COMPETITIVE athlete Q2066131
  cache HIT: COMPETITIVE__athlete.json
  returned: 18755
  exact-DOB resolved: 17 | cumulative Competitive supply: 27
supplement query: COMPETITIVE tennis_player Q10833314
  returned: 9833
  exact-DOB resolved: 25 | cumulative Competitive supply: 46
supplement query: COMPETITIVE boxer Q11338576
  returned: 14332
  exact-DOB resolved: 29 | cumulative Competitive supply: 70
supplement query: COMPETITIVE basketball_player Q3665646
  returned: 25000
  exact-DOB resolved: 5 | cumulative Competitive supply: 75
Football still needed; switching to deterministic 5-year DOB shards.
supplement shard: association_football_player 1900 - 1904
  returned: 3811
  exact-DOB resolved: 3 | cumulative Competitive supply: 78
supplement shard: association_football_player 1905 -

,axis,fresh_exact_DOB_available,C2_target,sufficient
0,COMPETITIVE,102,80,True
1,PROJECT,745,100,True
2,STATUS,169,140,True


CONFIRM2 fresh exact-DOB supply gate: PASS


## 4. Deterministically freeze fresh C2 pool

In [12]:
# ================================================================
# 4. Deterministically freeze fresh C2 pool
#    R3: female-only supply recovery within ORIGINAL Competitive taxonomy
#
# IMPORTANT:
# - Cell 3 already passed total-supply gate.
# - Previous Cell 4 accumulated Competitive supply but failed before membership freeze.
# - Do NOT continue generic football shards.
# - Add only enough fresh PASS_EXACT_DOB female Competitive supply
#   using existing Competitive occupations with Wikidata P21=female.
#
# No events / pairability / chronology / astrology / Control.
# ================================================================

COMP_AXIS = "COMPETITIVE"
FEMALE_QID = "Q6581072"

# Existing V5 Competitive roles only.
# Start with roles whose prior broad query was supply-truncated / large.
FEMALE_SUPPLY_ROLES = [
    ("basketball_player", "Q3665646"),
    ("association_football_player", "Q937857"),
]

competitive_target = int(TARGET[COMP_AXIS])
competitive_female_min = int(np.ceil(competitive_target * FEMALE_MIN_SHARE))


def comp_stats(df):
    g = df[df.axis == COMP_AXIS].copy()
    return {
        "total": int(len(g)),
        "female": int(g.gender.map(is_female).sum()),
    }


stats_before = comp_stats(cand)

print("Competitive before female-only supplement:", stats_before)
print(
    "Frozen requirements:",
    {
        "total_min": competitive_target,
        "female_min": competitive_female_min,
    },
)

# ----------------------------------------------------------------
# Freeze this supply-only remedy BEFORE querying.
# ----------------------------------------------------------------

female_patch = {
    "version": "V5_CONFIRM2_COMPETITIVE_FEMALE_ONLY_SUPPLY_PATCH_V1",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "status": (
        "PREDECLARED_AFTER_COMPETITIVE_FEMALE_SUPPLY_SHORTAGE_"
        "BEFORE_FEMALE_ONLY_ROLE_QUERY"
    ),
    "observed_supply_fact": {
        "competitive_total": stats_before["total"],
        "competitive_female": stats_before["female"],
        "frozen_total_target": competitive_target,
        "frozen_female_min": competitive_female_min,
    },
    "remedy": (
        "Use P21=female only within already-approved original Competitive occupations. "
        "Stop at the first complete role query boundary where total and female supply "
        "both meet the frozen guardrails."
    ),
    "roles_in_fixed_order": [
        {"role_family": x[0], "role_qid": x[1]}
        for x in FEMALE_SUPPLY_ROLES
    ],
    "female_qid": FEMALE_QID,
    "identity_rule": (
        "Wikidata English normalized name + exact day-level P569 DOB "
        "must equal one unique frozen Rodden-AA birth row."
    ),
    "forbidden": {
        "events": True,
        "event_years": True,
        "pairability": True,
        "pair_gap": True,
        "chronology": True,
        "astrology": True,
        "control": True,
        "candidate_performance": True,
    },
    "quota_reduction_allowed": False,
    "female_guardrail_reduction_allowed": False,
}

FEMALE_PATCH_PATH = (
    OUT / "V5_CONFIRM2_COMPETITIVE_FEMALE_ONLY_SUPPLY_PATCH_DECISION.json"
)

json.dump(
    female_patch,
    open(FEMALE_PATCH_PATH, "w", encoding="utf-8"),
    ensure_ascii=False,
    indent=2,
)


# ----------------------------------------------------------------
# Female-only WDQS query.
# P21 is used solely to satisfy the already-frozen sampling guardrail.
# ----------------------------------------------------------------

ENDPOINT_R3 = "https://query.wikidata.org/sparql"


def female_role_query(role_qid, limit=25000):
    return """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?person ?personLabel ?dob WHERE {
  ?person
      wdt:P106 wd:%s ;
      wdt:P21 wd:%s ;
      wdt:P569 ?dob ;
      rdfs:label ?personLabel .

  FILTER(LANG(?personLabel) = "en")
  FILTER(YEAR(?dob) >= 1900 && YEAR(?dob) <= 1995)
}
ORDER BY ?person
LIMIT %d
""" % (role_qid, FEMALE_QID, int(limit))


def run_wdqs_r3(query, cache_path, max_attempts=5):
    if cache_path.exists():
        print("  cache HIT:", cache_path.name)
        return json.load(open(cache_path, encoding="utf-8"))

    params = urllib.parse.urlencode({"query": query, "format": "json"})
    url = ENDPOINT_R3 + "?" + params

    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": (
            "Chartpalja-Saju-Research/1.0 "
            "(CONFIRM2 female sampling guardrail; no outcome query)"
        ),
    }

    last = None
    for attempt in range(max_attempts):
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as resp:
                obj = json.loads(resp.read().decode("utf-8"))

            json.dump(
                obj,
                open(cache_path, "w", encoding="utf-8"),
                ensure_ascii=False,
            )
            return obj

        except Exception as e:
            last = e
            if attempt + 1 < max_attempts:
                sleep_s = 3 * (2 ** attempt)
                print("  WDQS retry", sleep_s, "sec:", repr(e))
                time.sleep(sleep_s)

    raise RuntimeError(
        f"Female-only WDQS failed for {cache_path.name}: {last!r}"
    )


def resolve_female_role(role_name, role_qid):
    cache_path = CACHE / f"COMPETITIVE__FEMALE__{role_name}.json"

    print("\nfemale-only supplement:", role_name, role_qid)

    obj = run_wdqs_r3(
        female_role_query(role_qid),
        cache_path,
    )

    bindings = (
        obj.get("results", {})
        .get("bindings", [])
    )

    print("  returned:", len(bindings))

    rows = []

    for b in bindings:
        uri = (b.get("person") or {}).get("value", "")
        label = (b.get("personLabel") or {}).get("value", "")
        dob = (b.get("dob") or {}).get("value", "")

        qid_person = uri.rsplit("/", 1)[-1] if uri else ""

        m = re.match(
            r"^(\d{4})-(\d{2})-(\d{2})",
            dob,
        )

        if not (qid_person and label and m):
            continue

        yyyy, mo, dd = map(int, m.groups())

        if mo < 1 or dd < 1:
            continue

        rows.append(
            {
                "axis": COMP_AXIS,
                "role_family": role_name,
                "role_qid": role_qid,
                "wikidata_id": qid_person,
                "name": label,
                "norm_name_c2": norm_name(label),
                "wikidata_birth_date": "%04d-%02d-%02d" % (yyyy, mo, dd),
            }
        )

    if not rows:
        return pd.DataFrame(), {
            "role_family": role_name,
            "role_qid": role_qid,
            "returned_rows": len(bindings),
            "resolved_exact_DOB": 0,
            "resolved_female": 0,
            "cache_sha256": sha256_file(cache_path),
        }

    raw = pd.DataFrame(rows)

    raw = (
        raw.sort_values(["wikidata_id", "name"])
        .drop_duplicates(
            ["wikidata_id", "norm_name_c2", "wikidata_birth_date"]
        )
        .reset_index(drop=True)
    )

    # Exact name + exact DOB against frozen unique Rodden-AA rows.
    resolved = raw.merge(
        aa_unique,
        left_on=["norm_name_c2", "wikidata_birth_date"],
        right_on=["norm_name_c2", "birth_date"],
        how="inner",
        validate="many_to_one",
    )

    # Exclude every prior DEV/CONFIRM research identity.
    resolved = resolved[
        ~resolved.norm_name_c2.isin(used_names)
        & ~resolved.wikidata_id.astype(str).isin(used_qids)
    ].copy()

    # Exclude identities already accumulated in current C2 candidate pool.
    current_qids = set(cand.wikidata_id.dropna().astype(str))
    current_names = set(cand.name.dropna().map(norm_name))

    resolved = resolved[
        ~resolved.wikidata_id.astype(str).isin(current_qids)
        & ~resolved.norm_name_c2.isin(current_names)
    ].copy()

    resolved = (
        resolved.sort_values(["wikidata_id", "source_row_key"])
        .drop_duplicates("wikidata_id")
        .drop_duplicates("norm_name_c2")
        .reset_index(drop=True)
    )

    # The birth snapshot must also identify these as female.
    resolved = resolved[
        resolved.gender.map(is_female)
    ].copy()

    resolved["linkage_status"] = "PASS_EXACT_DOB"
    resolved["candidate_source_c2"] = (
        "C2_COMPETITIVE_FEMALE_ONLY_ORIGINAL_ROLE_EXACT_DOB"
    )
    resolved["rodden_rating"] = "AA"
    resolved["birth_source"] = "Frozen VedAstro PersonList-15k snapshot"
    resolved["selection_information_used"] = (
        "ORIGINAL_COMPETITIVE_ROLE + P21_FEMALE_SAMPLING_GUARDRAIL + "
        "WIKIDATA_P569_EXACT_DOB_LINKAGE"
    )
    resolved["event_collection_started"] = False
    resolved["astrology_scored"] = False

    info = {
        "role_family": role_name,
        "role_qid": role_qid,
        "returned_rows": len(bindings),
        "resolved_exact_DOB": int(len(resolved)),
        "resolved_female": int(resolved.gender.map(is_female).sum()),
        "cache_sha256": sha256_file(cache_path),
    }

    return resolved, info


# ----------------------------------------------------------------
# Fixed role order. Stop only after a COMPLETE role query.
# ----------------------------------------------------------------

query_manifest = []

for role_name, role_qid in FEMALE_SUPPLY_ROLES:

    current = comp_stats(cand)

    if (
        current["total"] >= competitive_target
        and current["female"] >= competitive_female_min
    ):
        break

    extra, info = resolve_female_role(
        role_name,
        role_qid,
    )

    query_manifest.append(info)

    if len(extra):
        cand = align_append(cand, extra)

        cand["norm_name_c2"] = cand.name.map(norm_name)

        cand = (
            cand.sort_values(
                [
                    "axis",
                    "wikidata_id",
                    "norm_name_c2",
                    "candidate_source_c2",
                ],
                kind="stable",
            )
            .drop_duplicates("wikidata_id")
            .drop_duplicates(["axis", "norm_name_c2"])
            .reset_index(drop=True)
        )

    current = comp_stats(cand)

    print(
        "  new exact-DOB female:",
        info["resolved_female"],
        "| cumulative Competitive:",
        current["total"],
        "| cumulative female:",
        current["female"],
    )


stats_after = comp_stats(cand)

female_manifest = {
    "version": "V5_CONFIRM2_COMPETITIVE_FEMALE_ONLY_SUPPLY_MANIFEST_V1",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "status": (
        "CONFIRM2_COMPETITIVE_FEMALE_SUPPLY_PASS"
        if (
            stats_after["total"] >= competitive_target
            and stats_after["female"] >= competitive_female_min
        )
        else "CONFIRM2_COMPETITIVE_FEMALE_SUPPLY_STILL_INSUFFICIENT"
    ),
    "before": stats_before,
    "after": stats_after,
    "requirements": {
        "total_min": competitive_target,
        "female_min": competitive_female_min,
    },
    "queries": query_manifest,
    "female_supply_patch_sha256": sha256_file(FEMALE_PATCH_PATH),
    "events_used": False,
    "pairability_used": False,
    "chronology_used": False,
    "astrology_used": False,
    "control_used": False,
}

json.dump(
    female_manifest,
    open(
        OUT / "V5_CONFIRM2_COMPETITIVE_FEMALE_ONLY_SUPPLY_MANIFEST.json",
        "w",
        encoding="utf-8",
    ),
    ensure_ascii=False,
    indent=2,
)


if (
    stats_after["total"] < competitive_target
    or stats_after["female"] < competitive_female_min
):
    raise RuntimeError(
        {
            "status": (
                "V5_CONFIRM2_COMPETITIVE_FEMALE_SUPPLY_STILL_INSUFFICIENT"
            ),
            "competitive_supply": stats_after,
            "required_total": competitive_target,
            "required_female": competitive_female_min,
            "scores_viewed": False,
            "next_rule": (
                "Do not lower guardrails. "
                "Freeze a broader outcome-blind Competitive female role taxonomy "
                "before any further candidate query."
            ),
        }
    )


print()
print("Competitive female supply PASS:", stats_after)


# ================================================================
# Membership freeze only AFTER every axis passes total + female supply.
# ================================================================

def choose_axis(pool, axis, n):

    g = pool[
        pool.axis == axis
    ].copy()

    assert (
        g.linkage_status
        == "PASS_EXACT_DOB"
    ).all()

    g["c2_det_key"] = g.apply(
        lambda r: det_key(
            axis,
            str(r.wikidata_id),
            r["name"],
        ),
        axis=1,
    )

    g = (
        g.sort_values(
            ["c2_det_key", "wikidata_id", "norm_name_c2"]
        )
        .reset_index(drop=True)
    )

    females = g[
        g.gender.map(is_female)
    ].copy()

    need_f = int(
        np.ceil(
            n * FEMALE_MIN_SHARE
        )
    )

    if len(females) < need_f:
        raise RuntimeError(
            f"{axis}: female exact-DOB supply "
            f"{len(females)} < guardrail {need_f}"
        )

    chosen_f = females.head(need_f)

    chosen_f_qids = set(
        chosen_f.wikidata_id.astype(str)
    )

    remaining = g[
        ~g.wikidata_id.astype(str).isin(chosen_f_qids)
    ].copy()

    chosen = pd.concat(
        [
            chosen_f,
            remaining.head(n - need_f),
        ],
        ignore_index=True,
    )

    if len(chosen) != n:
        raise RuntimeError(
            f"{axis}: selected {len(chosen)}/{n}"
        )

    assert chosen.wikidata_id.nunique() == n
    assert chosen.gender.map(is_female).sum() >= need_f

    return chosen


# ----------------------------------------------------------------
# Hard supply audit for all axes BEFORE selection.
# ----------------------------------------------------------------

preselect_rows = []

for axis in AXES:

    g = cand[
        cand.axis == axis
    ]

    target_n = int(TARGET[axis])

    female_need = int(
        np.ceil(
            target_n * FEMALE_MIN_SHARE
        )
    )

    preselect_rows.append(
        {
            "axis": axis,
            "available": int(len(g)),
            "target": target_n,
            "female_available": int(
                g.gender.map(is_female).sum()
            ),
            "female_required": female_need,
            "total_ok": int(len(g)) >= target_n,
            "female_ok": int(
                g.gender.map(is_female).sum()
            ) >= female_need,
        }
    )


preselect_supply = pd.DataFrame(preselect_rows)

display(preselect_supply)

assert preselect_supply["total_ok"].all(), preselect_supply
assert preselect_supply["female_ok"].all(), preselect_supply


# ----------------------------------------------------------------
# Deterministic membership selection.
# ----------------------------------------------------------------

parts = [
    choose_axis(
        cand,
        axis,
        TARGET[axis],
    )
    for axis in AXES
]

c2 = pd.concat(
    parts,
    ignore_index=True,
)

assert len(c2) == 320
assert c2.wikidata_id.nunique() == 320

assert not set(c2.norm_name_c2) & used_names
assert not set(c2.wikidata_id.astype(str)) & used_qids

assert (
    c2.axis.value_counts().to_dict()
    == {
        "STATUS": 140,
        "PROJECT": 100,
        "COMPETITIVE": 80,
    }
)

c2 = (
    c2.sort_values(
        ["axis", "c2_det_key", "wikidata_id"]
    )
    .reset_index(drop=True)
)

c2["subject_id"] = [
    f"V5C2_{i+1:03d}"
    for i in range(len(c2))
]

c2["preassigned_axis"] = c2["axis"]
c2["split"] = "CONFIRM2"
c2["rodden_rating"] = "AA"
c2["event_collection_started"] = False
c2["astrology_scored"] = False

ROSTER = (
    OUT / "V5_CONFIRM2_SUBJECT_ROSTER_320_FROZEN.csv"
)

c2.to_csv(
    ROSTER,
    index=False,
)

print()
print("Fresh C2 roster frozen:", len(c2))

display(
    c2.groupby("preassigned_axis")
    .agg(
        n=("subject_id", "size"),
        female=(
            "gender",
            lambda s: s.map(is_female).sum(),
        ),
    )
)

Competitive before female-only supplement: {'total': 423, 'female': 12}
Frozen requirements: {'total_min': 80, 'female_min': 16}

female-only supplement: basketball_player Q3665646
  returned: 24488
  new exact-DOB female: 8 | cumulative Competitive: 431 | cumulative female: 20

Competitive female supply PASS: {'total': 431, 'female': 20}


,axis,available,target,female_available,female_required,total_ok,female_ok
0,COMPETITIVE,431,80,20,16,True,True
1,PROJECT,745,100,272,20,True,True
2,STATUS,169,140,16,28,True,False


AssertionError:           axis  available  target  female_available  female_required  \
0  COMPETITIVE        431      80                20               16   
1      PROJECT        745     100               272               20   
2       STATUS        169     140                16               28   

   total_ok  female_ok  
0      True       True  
1      True       True  
2      True      False  

## 5. Freeze 16 deterministic batches preserving original axis mixture

In [ ]:

# Pattern totals exactly C80 / P100 / S140.
batch_quota={}
for b in range(1,17):
    if b in {1,5,9,13}:
        batch_quota[b]={"COMPETITIVE":5,"PROJECT":7,"STATUS":8}
    else:
        batch_quota[b]={"COMPETITIVE":5,"PROJECT":6,"STATUS":9}

assert sum(x["COMPETITIVE"] for x in batch_quota.values())==80
assert sum(x["PROJECT"] for x in batch_quota.values())==100
assert sum(x["STATUS"] for x in batch_quota.values())==140

axis_lists={
    a:c2[c2.preassigned_axis==a].sort_values(["c2_det_key","subject_id"]).reset_index(drop=True)
    for a in AXES
}
cursor={a:0 for a in AXES}
rows=[]

for b in range(1,17):
    temp=[]
    for a in AXES:
        n=batch_quota[b][a]
        start=cursor[a]; end=start+n
        block=axis_lists[a].iloc[start:end].copy()
        if len(block)!=n: raise RuntimeError((b,a,len(block),n))
        cursor[a]=end
        temp.append(block)

    z=pd.concat(temp,ignore_index=True)
    # Within-batch order is deterministic but mixed across axes.
    z["batch_order_key"]=z.apply(
        lambda r:hashlib.sha256(
            f"{SEED}|B{b:02d}|{r.subject_id}|{r.wikidata_id}".encode()
        ).hexdigest(),axis=1
    )
    z=z.sort_values(["batch_order_key","subject_id"]).reset_index(drop=True)
    z["batch_id"]=b
    z["research_order_in_batch"]=np.arange(1,len(z)+1)
    rows.append(z)

master=pd.concat(rows,ignore_index=True)
assert len(master)==320 and master.subject_id.nunique()==320
assert master.groupby("batch_id").size().eq(20).all()

# Full master retains birth lineage locally.
MASTER=OUT/"V5_CONFIRM2_MASTER_WORKLIST_16BATCH.csv"
master.to_csv(MASTER,index=False)

# Research worklists deliberately omit birth time/coordinates and astrology-capable fields.
research_cols=[
    "batch_id","research_order_in_batch","subject_id","name","preassigned_axis",
    "birth_date","birth_place","wikidata_id","role_family","role_qid",
    "candidate_source_c2","linkage_status"
]
for b in range(1,17):
    w=master[master.batch_id==b][research_cols].copy()
    path=BATCH/f"V5_CONFIRM2_BATCH_{b:02d}_SUBJECT_WORKLIST.csv"
    w.to_csv(path,index=False)

display(
    master.groupby(["batch_id","preassigned_axis"]).size()
    .unstack(fill_value=0)
)


## 6. Freeze decision + adaptive stopping contract

In [ ]:

batch_hashes={
    f"{b:02d}":sha256_file(BATCH/f"V5_CONFIRM2_BATCH_{b:02d}_SUBJECT_WORKLIST.csv")
    for b in range(1,17)
}

decision={
    "version":"V5_CONFIRM2_POOL_FREEZE_DECISION_V1",
    "notebook_version":NOTEBOOK_VERSION,
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":"V5_CONFIRM2_POOL_320_AND_16_BATCHES_FROZEN_READY_FOR_BLIND_EVENT_COLLECTION",
    "confirm1":{
        "status":c1["status"],
        "classification":c1["classification"],
        "scoring_performed":False,
        "primary_pairable_subjects":8,
        "frozen_event_corpus_remains_unchanged":True
    },
    "confirm2":{
        "n":320,
        "axis_counts":c2.preassigned_axis.value_counts().to_dict(),
        "roster_sha256":sha256_file(ROSTER),
        "master_worklist_sha256":sha256_file(MASTER),
        "batch_worklist_sha256":batch_hashes,
        "identity_rule":"PASS_EXACT_DOB only",
        "all_prior_DEV_CONFIRM_excluded":True,
        "membership_used_event_outcomes":False,
        "membership_used_pairability":False,
        "membership_used_chronology":False,
        "membership_used_astrology":False,
        "membership_used_control":False
    },
    "adaptive_stopping":{
        "research_batches_in_order":list(range(1,17)),
        "check_only_after_complete_batch_event_freeze":True,
        "stop_when_C1_plus_C2_cumulative_PRIMARY_pairable_subjects_ge":30,
        "C1_starting_PRIMARY_pairable_subjects":8,
        "only_pairability_count_may_trigger_stop":True,
        "V5_or_Control_scores_before_stop_forbidden":True,
        "chronology_direction_before_stop_forbidden":True,
        "after_stop":"Run frozen C1+C2 combined one-shot final scoring under unchanged 27A confirmation gates."
    },
    "lineage":{
        "confirm2_protocol_sha256":sha256_file(C2_PROTOCOL),
        "candidate_coefficients_sha256":sha256_file(CAND_COEF),
        "saju_engine_py_sha256":sha256_file(ROOT/"saju_engine.py"),
        "confirm1_final_decision_sha256":sha256_file(C1_FINAL),
        "confirm1_event_freeze_sha256":sha256_file(C1_FREEZE),
        "candidate_pool_sha256":sha256_file(POOL),
        "birth_snapshot_sha256":sha256_file(BIRTH)
    },
    "next_rule":(
        "Research CONFIRM2 Batch 01 only under the same blind event contract. "
        "After its event corpus is frozen, recompute cumulative PRIMARY pairability using C1 + completed C2 batches only. "
        "If <30, proceed to Batch 02; if >=30, stop event research and perform the final one-shot evaluation."
    )
}
DEC=OUT/"V5_CONFIRM2_POOL_FREEZE_DECISION.json"
json.dump(decision,open(DEC,"w",encoding="utf-8"),ensure_ascii=False,indent=2)
print(json.dumps({
    "status":decision["status"],
    "n":320,
    "axis_counts":decision["confirm2"]["axis_counts"],
    "first_next_action":"CONFIRM2 Batch 01 blind event research only"
},ensure_ascii=False,indent=2))



## Send back after Run All

Send these 3 files first:

```text
V5_CONFIRM2_POOL_FREEZE_DECISION.json
V5_CONFIRM2_SUBJECT_ROSTER_320_FROZEN.csv
V5_CONFIRM2_BATCH_01_SUBJECT_WORKLIST.csv
```

Do not research Batch 02 yet.

I will then run the same blind source sweep for Batch 01 and return its event/audit/manifest files.
After each completed batch, a lightweight cumulative-pairability gate decides whether another batch is necessary.
